# 03 — Silver

Typed, filtered, deduplicated. Source vocabulary becomes project vocabulary:
`Region`, `Drivmedel` and `Tid` become `municipality_code`, `fuel_type_code`,
`year` and `month`.

What this layer does:

- Filters Region to 4-character municipality codes. National and county rows
  are aggregates of these and would double-count in a `SUM`.
- Drops ownership categories `050` and `060` - ratios per 1,000 inhabitants,
  not counts, and they must not sit in a measure column.
- Drops region code `1917` (Heby, pre-2007 code, no data in our period).
- Aggregates hourly and 15-minute prices to one row per price area and day.
- Maps municipality to price area through a maintained county-level seed.

All loads use `MERGE` on the business key, so a re-run updates in place rather
than appending.

In [0]:
from pyspark.sql import functions, Window

CATALOG = "axenil_assignment1"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")


def merge_into_silver(source_dataframe, table_name, key_columns, order_column="_ingested_at"):
    """Deduplicate the source on the business key, then MERGE into the target.

    MERGE protects the target across runs but does not deduplicate the source,
    so a duplicate business key in bronze would insert twice on a first load.
    """
    target = f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"

    if order_column in source_dataframe.columns:
        newest_first = Window.partitionBy(*key_columns).orderBy(
            functions.col(order_column).desc()
        )
        source_dataframe = (
            source_dataframe
            .withColumn("_row_number", functions.row_number().over(newest_first))
            .filter(functions.col("_row_number") == 1)
            .drop("_row_number", order_column)
        )

    source_dataframe.createOrReplaceTempView("merge_source")

    if not spark.catalog.tableExists(target):
        source_dataframe.limit(0).write.saveAsTable(target)

    condition = " AND ".join(f"target.{column} = source.{column}" for column in key_columns)
    spark.sql(f"""
        MERGE INTO {target} AS target
        USING merge_source AS source
          ON {condition}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"{table_name}: {spark.table(target).count()} rows")

In [0]:
# Mapping from county to electricity price area.
# Source: Svenska kraftnät price area map, read 2026-09-24.
# Rule: each county is assigned the price area covering most of its area.
# Counties split by the SE2/SE3 and SE3/SE4 boundaries are flagged below —
# Kalmar for example has roughly 30 % of its area in SE3 while mapped to SE4.
COUNTY_TO_PRICE_AREA = [
    ("25", "SE1"), ("24", "SE2"), ("23", "SE2"), ("22", "SE2"), ("21", "SE2"),
    ("01", "SE3"), ("03", "SE3"), ("04", "SE3"), ("05", "SE3"), ("06", "SE3"),
    ("09", "SE3"), ("14", "SE3"), ("17", "SE3"), ("18", "SE3"), ("19", "SE3"),
    ("20", "SE3"),
    ("07", "SE4"), ("08", "SE4"), ("10", "SE4"), ("12", "SE4"), ("13", "SE4"),
]
SPLIT_COUNTIES = ["07", "08", "13", "20", "21"]

# 1917 = Heby, pre-2007 code before the county change (now 0331).
DISCONTINUED_REGION_CODES = ["1917"]

# 050 and 060 are ratios per 1 000 inhabitants, not counts.
RATIO_OWNERSHIP_CODES = ["050", "060"]

region_labels = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.scb_new_registrations_labels")
         .filter(functions.col("dimension_id") == "Region")
)

municipalities = (
    region_labels
    .filter(functions.length("code") == 4)
    .filter(~functions.col("code").isin(DISCONTINUED_REGION_CODES))
    .select(
        functions.col("code").alias("municipality_code"),
        functions.col("label").alias("municipality_name"),
        functions.substring("code", 1, 2).alias("county_code"),
    )
)

counties = (
    region_labels
    .filter(functions.length("code") == 2)
    .select(
        functions.col("code").alias("county_code"),
        functions.col("label").alias("county_name"),
    )
)

price_areas = spark.createDataFrame(COUNTY_TO_PRICE_AREA, ["county_code", "price_area"])

municipality = (
    municipalities
    .join(counties, on="county_code", how="left")
    .join(price_areas, on="county_code", how="left")
    .withColumn("price_area_is_split", functions.col("county_code").isin(SPLIT_COUNTIES))
    .select("municipality_code", "municipality_name", "county_code",
            "county_name", "price_area", "price_area_is_split")
)

merge_into_silver(municipality, "municipality", ["municipality_code"])

In [0]:
fuel_type = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.scb_new_registrations_labels")
    .filter(functions.col("dimension_id") == "Drivmedel")
    .select(
        functions.col("code").alias("fuel_type_code"),
        functions.col("label").alias("fuel_type_name"),
    )
    .withColumn("is_electrified", functions.col("fuel_type_code").isin(["120", "140"]))
)
merge_into_silver(fuel_type, "fuel_type", ["fuel_type_code"])

ownership_category = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.scb_cars_in_traffic_labels")
    .filter(functions.col("dimension_id") == "Agarkategori")
    .filter(~functions.col("code").isin(RATIO_OWNERSHIP_CODES))
    .select(
        functions.col("code").alias("ownership_category_code"),
        functions.col("label").alias("ownership_category_name"),
    )
)
merge_into_silver(ownership_category, "ownership_category", ["ownership_category_code"])

In [0]:
new_registrations = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.scb_new_registrations")
    .filter(functions.length("Region") == 4)
    .filter(~functions.col("Region").isin(DISCONTINUED_REGION_CODES))
    .select(
        functions.col("Region").alias("municipality_code"),
        functions.substring("Tid", 1, 4).cast("int").alias("year"),
        functions.substring("Tid", 6, 2).cast("int").alias("month"),
        functions.col("Drivmedel").alias("fuel_type_code"),
        functions.col("value").cast("int").alias("registration_count"),
        functions.col("_ingested_at"),
    )
)

merge_into_silver(
    new_registrations,
    "new_registrations",
    ["municipality_code", "year", "month", "fuel_type_code"],
)

In [0]:
cars_in_traffic = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.scb_cars_in_traffic")
    .filter(functions.length("Region") == 4)
    .filter(~functions.col("Region").isin(DISCONTINUED_REGION_CODES))
    .filter(~functions.col("Agarkategori").isin(RATIO_OWNERSHIP_CODES))
    .select(
        functions.col("Region").alias("municipality_code"),
        functions.col("Tid").cast("int").alias("year"),
        functions.col("Agarkategori").alias("ownership_category_code"),
        functions.col("value").cast("int").alias("vehicle_count"),
        functions.col("_ingested_at"),
    )
)

merge_into_silver(
    cars_in_traffic,
    "cars_in_traffic",
    ["municipality_code", "year", "ownership_category_code"],
)

In [0]:
daily_prices = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.electricity_prices")
    .select(
        functions.col("elomrade").alias("price_area"),
        # local date as published — avoids timezone conversion entirely
        functions.to_date(functions.substring("time_start", 1, 10)).alias("price_date"),
        functions.col("SEK_per_kWh").alias("sek_per_kwh"),
    )
    .groupBy("price_area", "price_date")
    .agg(
        functions.avg("sek_per_kwh").alias("avg_sek_per_kwh"),
        functions.min("sek_per_kwh").alias("min_sek_per_kwh"),
        functions.max("sek_per_kwh").alias("max_sek_per_kwh"),
        functions.count("*").alias("period_count"),
    )
    .withColumn(
        "resolution_minutes",
        functions.when(functions.col("period_count") > 30, 15).otherwise(60),
    )
)

merge_into_silver(daily_prices, "electricity_prices_daily", ["price_area", "price_date"])